In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=8a930f95796a372757d78b459f6c57e716174cc1e04ae9de57780d8286ffed90
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# ============================================================
# SHARED UTILITIES
# ============================================================

simulator = BasicSimulator()

def quantum_random_bit():
    """
    Generate a single random bit (0 or 1) by placing a qubit in
    superposition |+> = H|0> and measuring it.
    This is a true quantum random number, not a classical PRNG.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0> -> |+> = (|0> + |1>) / sqrt(2)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    """Generate a list of n random bits using quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]

# Number of qubits to send in the protocol
N_QUBITS = 100

# Fraction of sifted key sacrificed for error checking
SAMPLE_FRACTION = 0.2

# Error rate threshold above which an attack is declared
ERROR_THRESHOLD = 0.1  # 10%

print("Utilities ready. Simulator:", simulator.name)

Utilities ready. Simulator: basic_simulator


In [4]:
# ============================================================
# ALICE
# ============================================================

def alice_encode(bit, basis):
    """
    Alice encodes a single bit into a qubit circuit.
    basis 0 = rectilinear (+): encode as |0> or |1>
    basis 1 = diagonal   (x): encode as |+> or |->
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)       # |0> -> |1>
    if basis == 1:
        qc.h(0)       # |0> -> |+>  or  |1> -> |->
    return qc

# Alice generates her random bits and bases
print("Alice: generating random bits and bases...")
alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)

# Alice encodes each bit into a qubit circuit and "sends" them
alice_qubits = [alice_encode(alice_bits[i], alice_bases[i]) for i in range(N_QUBITS)]

print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}  (0=+, 1=x)")

Alice: generating random bits and bases...
Alice bits  (first 20): [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1]
Alice bases (first 20): [0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1]  (0=+, 1=x)


In [5]:
# ============================================================
# EVE  (attacker — intercept-resend attack)
# ============================================================

def eve_intercept(qubit_circuit):
    """
    Eve intercepts a qubit from Alice.
    1. She picks a random basis (she has no idea which Alice used).
    2. She measures the qubit in her chosen basis — this irreversibly
       collapses the quantum state if her basis is wrong.
    3. She re-encodes her measurement result in her chosen basis
       and forwards the new qubit to Bob.

    Returns: (new qubit circuit to forward, eve's basis, eve's measured bit)
    """
    eve_basis = quantum_random_bit()   # Eve randomly guesses a basis

    # Eve measures the qubit in her chosen basis
    qc_measure = qubit_circuit.copy()
    if eve_basis == 1:
        qc_measure.h(0)   # rotate to diagonal basis before measuring
    qc_measure.measure(0, 0)

    job = simulator.run(transpile(qc_measure, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    eve_bit = int(list(counts.keys())[0])

    # Eve re-encodes her result and forwards a fresh qubit to Bob
    qc_forward = QuantumCircuit(1, 1)
    if eve_bit == 1:
        qc_forward.x(0)
    if eve_basis == 1:
        qc_forward.h(0)

    return qc_forward, eve_basis, eve_bit

# Eve intercepts all qubits in transit
print("Eve: intercepting all qubits...")
forwarded_qubits = []
eve_bases = []
eve_bits  = []

for i in range(N_QUBITS):
    fwd, eb, ebit = eve_intercept(alice_qubits[i])
    forwarded_qubits.append(fwd)
    eve_bases.append(eb)
    eve_bits.append(ebit)

# How often did Eve guess the correct basis?
correct_guesses = sum(1 for i in range(N_QUBITS) if eve_bases[i] == alice_bases[i])
print(f"Eve guessed correct basis : {correct_guesses}/{N_QUBITS} times ({correct_guesses/N_QUBITS*100:.1f}%)")
print(f"Eve bases   (first 20): {eve_bases[:20]}")
print(f"Eve bits    (first 20): {eve_bits[:20]}")

Eve: intercepting all qubits...
Eve guessed correct basis : 44/100 times (44.0%)
Eve bases   (first 20): [1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]
Eve bits    (first 20): [0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]


In [6]:
# ============================================================
# BOB
# ============================================================

def bob_measure(qubit_circuit, basis):
    """
    Bob measures the qubit he received (from Eve, unknowingly).
    basis 0 = rectilinear (+): measure directly
    basis 1 = diagonal   (x): apply H first, then measure
    """
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)       # rotate back from diagonal basis
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Bob measures the forwarded qubits (which Eve tampered with)
print("Bob: generating random bases and measuring (Eve's forwarded) qubits...")
bob_bases   = quantum_random_bits(N_QUBITS)
bob_results = [bob_measure(forwarded_qubits[i], bob_bases[i]) for i in range(N_QUBITS)]

print(f"Bob bases   (first 20): {bob_bases[:20]}  (0=+, 1=x)")
print(f"Bob results (first 20): {bob_results[:20]}")

Bob: generating random bases and measuring (Eve's forwarded) qubits...
Bob bases   (first 20): [0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1]  (0=+, 1=x)
Bob results (first 20): [1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0]


In [7]:
# ============================================================
# BASIS SIFTING  (classical communication between Alice & Bob)
# ============================================================

alice_sifted = []
bob_sifted   = []

for i in range(N_QUBITS):
    if alice_bases[i] == bob_bases[i]:   # bases matched -> keep this bit
        alice_sifted.append(alice_bits[i])
        bob_sifted.append(bob_results[i])

sifted_length = len(alice_sifted)
print(f"Qubits sent       : {N_QUBITS}")
print(f"Sifted key length : {sifted_length}  (~50% expected)")
print(f"\nAlice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted (first 20): {bob_sifted[:20]}")

# Show raw mismatch for transparency
raw_errors = sum(1 for a, b in zip(alice_sifted, bob_sifted) if a != b)
print(f"\nRaw mismatches in full sifted key: {raw_errors}/{sifted_length} ({raw_errors/sifted_length*100:.1f}%)")

Qubits sent       : 100
Sifted key length : 48  (~50% expected)

Alice sifted (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0]
Bob   sifted (first 20): [1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0]

Raw mismatches in full sifted key: 15/48 (31.2%)


In [8]:
# ============================================================
# ERROR CHECKING  (classical communication between Alice & Bob)
# ============================================================

sample_size = max(1, int(sifted_length * SAMPLE_FRACTION))

# Use quantum randomness to pick which indices to sample
sampled_indices = set()
while len(sampled_indices) < sample_size:
    bits_needed = max(1, math.ceil(math.log2(sifted_length)))
    idx = int("".join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
    if idx < sifted_length:
        sampled_indices.add(idx)

# Count mismatches in the sample
errors = sum(1 for i in sampled_indices if alice_sifted[i] != bob_sifted[i])
error_rate = errors / sample_size

print(f"Sample size : {sample_size} bits ({SAMPLE_FRACTION*100:.0f}% of sifted key)")
print(f"Errors      : {errors}")
print(f"Error rate  : {error_rate*100:.1f}%")
print(f"Threshold   : {ERROR_THRESHOLD*100:.0f}%")
print()

if error_rate > ERROR_THRESHOLD:
    print("ATTACK DETECTED — aborting key exchange!")
    print("The high error rate indicates an eavesdropper on the channel.")
    print("Alice and Bob discard the key and do not communicate further.")
else:
    print("No attack detected. Key exchange successful.")
    final_key_alice = [alice_sifted[i] for i in range(sifted_length) if i not in sampled_indices]
    final_key_bob   = [bob_sifted[i]   for i in range(sifted_length) if i not in sampled_indices]
    print(f"Final key length : {len(final_key_alice)} bits")
    print(f"Keys match       : {final_key_alice == final_key_bob}")

Sample size : 9 bits (20% of sifted key)
Errors      : 4
Error rate  : 44.4%
Threshold   : 10%

ATTACK DETECTED — aborting key exchange!
The high error rate indicates an eavesdropper on the channel.
Alice and Bob discard the key and do not communicate further.


In [9]:
# ============================================================
# ANALYSIS — What did Eve actually learn?
# ============================================================

# Among qubits where BOTH alice_bases == bob_bases (the sifted key bits),
# how many of those did Eve also guess correctly?
eve_knew = 0
sifted_count = 0

for i in range(N_QUBITS):
    if alice_bases[i] == bob_bases[i]:   # this bit ended up in the sifted key
        sifted_count += 1
        if eve_bases[i] == alice_bases[i]:   # Eve also used the correct basis
            eve_knew += 1

print(f"Sifted key bits           : {sifted_count}")
print(f"Bits Eve knew correctly   : {eve_knew} ({eve_knew/sifted_count*100:.1f}%)")
print(f"Bits Eve guessed randomly : {sifted_count - eve_knew} ({(sifted_count-eve_knew)/sifted_count*100:.1f}%)")
print()
print("Conclusion: Eve obtained partial information about the key,")
print("but her presence was revealed by the elevated error rate.")
print("BB84 guarantees: if the channel is accepted, Eve's information is negligible.")

Sifted key bits           : 48
Bits Eve knew correctly   : 21 (43.8%)
Bits Eve guessed randomly : 27 (56.2%)

Conclusion: Eve obtained partial information about the key,
but her presence was revealed by the elevated error rate.
BB84 guarantees: if the channel is accepted, Eve's information is negligible.
